In [1]:
pip install pandas sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.7 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.7 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.7 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.7 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.7 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.1/9.7 MB 229.7 kB/s eta 0:00:42
   ---------------------------------------- 0.1/9.7 MB 229.7 kB/s eta 0:00:42
   ---------------------------------------- 0.1/9.7 MB 201.8 kB/s eta 0:00:48
   ---------------------------------------- 0.1/9.7 MB 201.8 kB/s eta 0:00:48
   ---------------------------------------- 0.1/9.7 MB 211.6 kB/s eta 0:00:46
   ----

In [ ]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
import pandas as pd
from pathlib import Path
import time

import os
from dotenv import load_dotenv

PROJECT_ROOT = Path(__file__).parent.parent
load_dotenv(PROJECT_ROOT / ".env")

# =========================================================
# 🔧 CONFIGURATION
# =========================================================
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASS = os.getenv("MYSQL_PASS")
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))
MYSQL_DB   = os.getenv("MYSQL_DB", "ecommerce_retention")

# 🎯 YOUR CSV PATH
CSV_PATH = r"C:\Users\dell\Downloads\online+retail\online_retail_II_raw.csv"
# =========================================================

# 🎯 ENCODE PASSWORD (this fixes the @ issue)
safe_pass = quote_plus(MYSQL_PASS)

# 🎯 BUILD CONNECTION STRING WITH ENCODED PASSWORD
connection_string = f"mysql+pymysql://{MYSQL_USER}:{safe_pass}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"

print(f"📥 Reading {CSV_PATH}...")
df = pd.read_csv(CSV_PATH, dtype=str, encoding="latin1", low_memory=False)
print(f"   ✅ {len(df):,} rows loaded")
print(f"   Columns: {list(df.columns)}")

# Match to MySQL table
df.columns = ["Invoice", "StockCode", "Description", "Quantity",
              "InvoiceDate", "Price", "CustomerID", "Country"]

print(f"\n🔌 Connecting to MySQL ({MYSQL_HOST}/{MYSQL_DB})...")
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1")).scalar()
    print("   ✅ Connected")
except Exception as e:
    print(f"   ❌ Connection failed: {e}")
    print("   Check your password in MYSQL_PASS above.")
    raise SystemExit(1)

# Truncate
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE online_retail_II_raw"))
print("🗑️  Truncated existing table")

# Import
print(f"\n📤 Importing {len(df):,} rows (takes 3-8 minutes)...")
start = time.time()
df.to_sql(
    "online_retail_II_raw",
    engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi",
)
elapsed = time.time() - start
print(f"✅ Import complete in {elapsed/60:.1f} minutes")

# Verify
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM online_retail_II_raw")).scalar()
print(f"\n📊 Total rows in table: {count:,}")

📥 Reading C:\Users\dell\Downloads\online+retail\online_retail_II_raw.csv...
   ✅ 541,909 rows loaded
   Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']

🔌 Connecting to MySQL (localhost/ecommerce_retention)...
   ✅ Connected
🗑️  Truncated existing table

📤 Importing 541,909 rows (takes 3-8 minutes)...


C:\Users\dell\AppData\Local\Temp\ipykernel_16232\2126122001.py:56: UserWarning: The provided table name 'online_retail_II_raw' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


✅ Import complete in 4.7 minutes

📊 Total rows in table: 541,909


In [ ]:
"""
Export cleaned MySQL tables to CSV for Python analysis.
Run AFTER all 4 SQL scripts have finished.
"""
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

import os
from dotenv import load_dotenv

PROJECT_ROOT = Path(__file__).parent.parent
load_dotenv(PROJECT_ROOT / ".env")

MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASS = os.getenv("MYSQL_PASS")
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))
MYSQL_DB   = os.getenv("MYSQL_DB", "ecommerce_retention")

# Encode password (handles @ and special chars)
safe_pass = quote_plus(MYSQL_PASS)
conn_str = f"mysql+pymysql://{MYSQL_USER}:{safe_pass}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
engine = create_engine(conn_str)

# Output folder (creates it if missing)
OUT = Path(r"C:\Users\dell\Desktop\ECommerce_Retention_LTV_Project\Data\processed")
OUT.mkdir(parents=True, exist_ok=True)

tables = [
    "cleaned_data",
    "rfm_summary",
    "cohort_retention_matrix",
    "churn_modeling_data",
]

print("📤 Exporting tables to CSV...\n")

for t in tables:
    print(f"   Exporting {t}...")
    df = pd.read_sql(f"SELECT * FROM {t}", engine)
    filepath = OUT / f"{t}.csv"
    df.to_csv(filepath, index=False)
    print(f"   ✅ {len(df):,} rows → {filepath.name}")

print(f"\n🎉 Export complete! Files saved to:")
print(f"   {OUT.resolve()}")

📤 Exporting tables to CSV...

   Exporting cleaned_data...
   ✅ 397,880 rows → cleaned_data.csv
   Exporting rfm_summary...
   ✅ 4,338 rows → rfm_summary.csv
   Exporting cohort_retention_matrix...
   ✅ 90 rows → cohort_retention_matrix.csv
   Exporting churn_modeling_data...
   ✅ 3,370 rows → churn_modeling_data.csv

🎉 Export complete! Files saved to:
   C:\Users\dell\Desktop\ECommerce_Retention_LTV_Project\Data\processed
